<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Introduction to OpenAI Agents SDK on Databricks
#### Duration: 20 minutes
## Overview
In this demo, you will build a single Data Analyst agent from scratch using the OpenAI Agents SDK on Databricks. You will define a custom tool with `@function_tool`, create and test the agent, register tools to Unity Catalog for governance, and enable MLflow tracing to observe execution.

## Learning Objectives
By the end of this demo, you will be able to:
1. Configure the OpenAI Agents SDK to use Databricks model routing
2. Define custom tools using `@function_tool`
3. Build and test a single agent with tools
4. Register tools to Unity Catalog for governance
5. Enable MLflow autologging and inspect execution traces

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-5

Installing 12 package(s) with uv...
  pyyaml==6.0.3
  databricks-openai==0.17.1
  databricks-agents==1.12.0
  databricks-sdk==0.145.0
  openai==3.22.1
  openai-agents==0.22.3
  nest_asyncio==1.6.0
  databricks-mcp==0.9.2
  unitycatalog-ai[databricks]==0.3.2
  mlflow==3.16.1
  mlflow-skinny==3.16.1
  mlflow-tracing==3.16.1
Library installation complete.


Starting Demo Environment Setup
Using config: Includes/config/config-5.yaml

[0] Compute check passed: Serverless 6 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17579966_1791918748' found. Using it.

[2] Setting up schema...
  Catalog : labuser17579966_1791918748
  Schema  : agent_training

[3] Setting up Unity Gateway model service...
  Model service created: labuser17579966_1791918748.agent_training.agent-training-model → databricks-gpt-5-2

[4] Processing datasets...
  Using Vocareum Delta Share: airbnb
Refreshing environment

Demo Environment Setup Complete


Catalog,"labuser17579966_1791918748<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Schema,agent_trainingCopy
Table,"labuser17579966_1791918748.agent_training.sf_airbnb_listings<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.sf_airbnb_listings';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Model service,"labuser17579966_1791918748.agent_training.agent-training-model<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791918748.agent_training.agent-training-model';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"


### A1. Import Libraries

To get started with the OpenAI SDK, we have a few things that need to be setup:
1. Configure the OpenAI Agents SDK to use Databricks model routing **using Unity Gateway Model Services** using `AsyncDatabricksOpenAI(use_ai_gateway=True)`.
1. Import MCP components needed to connect to agent tools that have already been registered to UC.
1. Initialize the Databricks `WorkspaceClient`.
1. Set the default trace processor to MLflow.

Additional documentation and reading:
- [`set_default_openai_client`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_client)
- [`set_default_openai_api`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_api)
- [`set_trace_processors`](https://github.com/openai/openai-agents-python/blob/main/docs/tracing.md)

In [0]:
from agents import Agent, Runner, function_tool, set_default_openai_api, set_default_openai_client
from agents.tracing import set_trace_processors
from databricks_openai import AsyncDatabricksOpenAI
import logging

set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))
set_default_openai_api("chat_completions")
set_trace_processors([])  # Disable OpenAI trace processor; use MLflow instead
logging.getLogger("openai").setLevel(logging.WARNING)

print("OpenAI Agents SDK configured to use Databricks model routing.")

OpenAI Agents SDK configured to use Databricks model routing.


## B. Define the Data Analyst Agent

### B1. Define the `classify_price_tier` Tool

The `classify_price_tier` tool classifies an Airbnb listing's nightly price into one of four market tiers based on the SF pricing distribution:

| Tier | Price Range | Percentile Band |
|---|---|---|
| **Budget** | Under $100 | Bottom 25% |
| **Mid-Range** | $100 -- $174 | 25--50% |
| **Premium** | $175 -- $299 | 50--75% |
| **Luxury** | $300+ | Top 25% |

The `@function_tool` decorator registers this function as a tool that the agent can invoke. The SDK extracts the function signature and docstring to generate the tool schema that the LLM uses to decide when and how to call the tool.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
An example production workflow is: prototype with <code>@function_tool</code>, then register to Unity Catalog, then access via a Managed MCP server. This demo follows the first two steps of that progression. In Section C, we explicitly re-register the function to Unity Catalog for governance.
</p>
</div>
</div>
</div>

In [0]:
import json

@function_tool
def classify_price_tier(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25–50%",      "Around the SF market median."),
        (300,  "Premium",   "50–75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

print(f"Defined tool: {classify_price_tier.name}")

Defined tool: classify_price_tier


### B2. Create the Agent

With the tool defined, we create a single agent that uses 
- `classify_price_tier` to answer questions about SF Airbnb pricing
- `instructions` parameter shapes the agent's behavior and tells it when to use the tool
- A Unity Catalog model service, referenced by its fully qualified name `{catalog_name}.{schema_name}.agent-training-model`

This is accomplished with the `Agent` class. You can read more about `Agent` [here](https://github.com/openai/openai-agents-python/blob/main/src/agents/agent.py).

#### B2.1 Inspect and Load the Agent Configuration
We will load the agent configuration file. This is a recommended practice since any updates to the agents' system prompts, name, etc. live outside this notebook. These variables will be stored in the dictionary `agents`:
- `agents["X"]['name']`: The agent's name
- `agents["X"]['llm_endpoint']`: The name of the Unity Catalog **model service** the agent calls (`agent-training-model`). The next cell combines it with your catalog and schema into the service's fully qualified name.
- `agents["X"]['system_prompt']`: The agent's system prompt

Navigate to the folder **agent_config** and inspect the agent contents for **data_analyst_agent.yaml**

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Why a model service?</strong>
<p style="margin: 8px 0 0 0; color: #333;">
In this course, agents call a <strong>model service</strong> registered in your catalog and schema instead of calling a foundation model endpoint such as <code>databricks-gpt-5-2</code> directly. The classroom setup created the service <code>agent-training-model</code>, which routes requests through Unity Gateway to <code>databricks-gpt-5-2</code>. Because the service is a Unity Catalog object, access to it is governed with Unity Catalog permissions, and because agents reference the service rather than the model, the model behind it can change without changing agent code.
</p>
</div>

In [0]:
import yaml
from pathlib import Path

agent_files = {
    "da": "data_analyst_agent"
}

agents = {}
for prefix, filename in agent_files.items():
    cfg = yaml.safe_load(Path(f"./agent_config/{filename}.yaml").read_text())
    agents[prefix] = {
        "name": cfg["name"],
        "llm_endpoint": cfg["llm_endpoint"],
        "system_prompt": cfg["system_prompt"],
    }

In [0]:
name = agents["da"]["name"]
system_prompt = agents["da"]["system_prompt"]
llm_endpoint = f"{catalog_name}.{schema_name}.{agents["da"]["llm_endpoint"]}"

data_analyst_agent = Agent(
    name=name,
    instructions=system_prompt,
    model=llm_endpoint,
    tools=[classify_price_tier],
)

print(f"Created agent: {data_analyst_agent.name}")
print(f"  Model: {data_analyst_agent.model}")
print(f"  Tools: {[t.name for t in data_analyst_agent.tools]}")

Created agent: Data Analyst
  Model: labuser17579966_1791918748.agent_training.agent-training-model
  Tools: ['classify_price_tier']


### B3. Test the Agent

We send three progressively complex queries to validate that the agent correctly invokes the tool and interprets the results. Each call uses `await Runner.run()`, which is the execution interface. It executes the full agent loop: the LLM decides whether to call a tool, executes it if needed, and generates a final response.

1. The next cell will run a straightforward pricing question that requires a single tool call.

In [0]:
query_1 = "Is $150/night a good deal in SF?"
result_1 = await Runner.run(data_analyst_agent, query_1)
print(f"User:     {query_1}")
print(f"Response: {result_1.final_output}")

User:     Is $150/night a good deal in SF?
Response: $150/night in San Francisco is **Mid-Range**—roughly in the **25–50th percentile** of SF Airbnb prices (i.e., **around the market median**).

**Is it a good deal?** It’s **fair/typical**, not a standout bargain. It becomes a **good deal** if it’s in a prime neighborhood, has strong reviews, or includes extras (parking, full kitchen, in-unit laundry).


2. The next cell will run a direct tier classification at the high end of the price spectrum.

In [0]:
query_2 = "What tier would a $500/night luxury listing fall into?"
result_2 = await Runner.run(data_analyst_agent, query_2)
print(f"User:     {query_2}")
print(f"Response: {result_2.final_output}")

User:     What tier would a $500/night luxury listing fall into?
Response: A **$500/night** Airbnb in San Francisco falls into the **Luxury** tier.

- **Tier:** Luxury  
- **Market position:** **Top 25%** of SF listing prices  
- **Interpretation:** **Among the most expensive listings in SF**


3. The next cell will run multiple prices in a single request, testing whether the agent calls the tool multiple times.

In [0]:
query_3 = "Classify these prices: $80, $200, $350"
result_3 = await Runner.run(data_analyst_agent, query_3)
print(f"User:     {query_3}")
print(f"Response: {result_3.final_output}")

User:     Classify these prices: $80, $200, $350
Response: - **$80/night** → **Budget** (bottom **25%**) — well below SF average.  
- **$200/night** → **Premium** (**50–75%**) — above-average for SF.  
- **$350/night** → **Luxury** (top **25%**) — among the most expensive in SF.


## C. Register Tools to Unity Catalog

### C1. Register `classify_price_tier` as a Python UC Function

The inline `@function_tool` is local to this notebook session. To make the tool governed, discoverable, and reusable across agents and workspaces, we register it to Unity Catalog as a Python function using `DatabricksFunctionClient`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The function registered below is a <strong>plain Python function</strong> (no <code>@function_tool</code> decorator). The <code>DatabricksFunctionClient</code> inspects the function signature and docstring to create the UC function metadata. The logic is identical to the tool we prototyped above.
</p>
</div>
</div>
</div>

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

def classify_price_uc(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25–50%",      "Around the SF market median."),
        (300,  "Premium",   "50–75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

client = DatabricksFunctionClient()
function_info = client.create_python_function(
    func=classify_price_uc,
    catalog=catalog_name,
    schema=schema_name,
    replace=True,
)

print(f"Created UC function: {function_info.full_name}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-4514bf8e-c32c-4337-885d-e651516ba523/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Created UC function: labuser17579966_1791918748.agent_training.classify_price_uc


### C2. Verify Registered Functions

Confirm that the function is visible in the Unity Catalog schema, then call it directly to verify it returns the expected output.
1. Navigate to the **Catalog Explorer** on the left side menu and search for `classify_price_uc`
1. Run the next cell to test the output from calling the function from UC

In [0]:
# Call classify_price_tier directly to verify
result = spark.sql(f"SELECT {catalog_name}.{schema_name}.classify_price_uc(150.0) AS result")
display(result)

result
"{ ""price"": 150.0, ""tier"": ""Mid-Range"", ""percentile_band"": ""25\u201350%"", ""interpretation"": ""Around the SF market median."" }"


### C3. Access the UC Function via a Managed MCP Server

Databricks provides **Managed MCP servers** that automatically expose Unity Catalog functions as tools. Any agent can discover and call these functions through the MCP protocol without custom integration code.

The MCP server endpoint for UC functions follows this pattern:
`https://workspace/api/2.0/mcp/functions/catalog/schema`

With the OpenAI Agents SDK, you use `McpServer` from `databricks_openai.agents` to connect as shown in the next cell.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The <code>McpServer</code> class from <code>databricks_openai.agents</code> is an async context manager designed for the Databricks Apps runtime. In a notebook, we use <code>DatabricksMCPClient</code> from <code>databricks_mcp</code> to interact with the same MCP endpoint.
See: Managed MCP servers (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">GCP</a>)
</p>
</div>
</div>
</div>


In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
workspace_host = ws.config.host

# Point to the MCP server for our catalog/schema
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

# Discover available tools (nest_asyncio patches the event loop for DatabricksMCPClient)
tools = mcp_client.list_tools()
print(f"Discovered {len(tools)} UC function tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description}")

Discovered 1 UC function tool(s):
  - labuser17579966_1791918748__agent_training__classify_price_uc: Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for questions like: "Is $180/night a good deal in SF?", "What tier does this listing fall into?" Returns a JSON object with tier, percentile_band, and a short interpretation.


### C4. Call the UC Function Through MCP

Now we call the registered UC function through the MCP protocol, just as an agent would. The default tool name is of the form `catalogName__schemaName__toolName` as demonstrated in the next cell.

In [0]:
# Call the UC function via MCP
tool_name = f"{catalog_name}__{schema_name}__classify_price_uc"
result = mcp_client.call_tool(tool_name, {"price": 120.00})
print(f"Result: {''.join([c.text for c in result.content])}")

Result: {"is_truncated":false,"columns":["output"],"rows":[["{\n  \"price\": 120.0,\n  \"tier\": \"Mid-Range\",\n  \"percentile_band\": \"25\\u201350%\",\n  \"interpretation\": \"Around the SF market median.\"\n}"]]}


## D. Enable MLflow Tracing

### D1. Enable Autologging

MLflow autologging for the OpenAI integration automatically captures execution traces for every agent run by patching the OpenAI SDK so that every call to various methods automatically generates MLflow traces. Each trace records the full sequence of LLM calls, tool invocations, inputs, outputs, and latencies without requiring any code changes to the agent itself.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Autologging captures: LLM call spans (model, prompt, completion tokens), tool call spans (function name, arguments, return value), and the overall agent run span with total latency. All traces are stored in the MLflow experiment associated with this notebook.
</p>
</div>
</div>
</div>

In [0]:
import mlflow

mlflow.openai.autolog()

### D2. Registering Traces to Unity Catalog
At this point, our traces haven't been logged. This means we have no insight into things like token usage, execution time per tool call, etc. 

Next, we'll show how to register traces in Unity Catalog. Databricks recommends this pattern for new and production workloads. 

1. Instantiate the `UnityCatalog` class from `mlflow` and call it `uc`.

In [0]:
from mlflow.entities.trace_location import UnityCatalog

uc = UnityCatalog(
    catalog_name = catalog_name,
    schema_name = schema_name,
    table_prefix = "oai_sa" # oai_sa: "OpenAI Single Agent"
)

2. Create and set the experiment location using `mlflow.set_experiment()`. In the code below, we will use `dbutils`, which is a Databricks utility library that allows users to do things like get the current notebook path within the workspace.

In [0]:
experiment_path = f"/Users/{username}/oai-sa-sdk-training"

mlflow.set_experiment(
    experiment_name=experiment_path,
    trace_location=uc,
)

2026/10/13 21:32:41 INFO mlflow.tracking.fluent: Experiment with name '/Users/labuser17579966_1791918748@vocareum.com/oai-sa-sdk-training' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3585981954521660', creation_time=1791927161610, effective_trace_archival_retention=None, experiment_id='3585981954521660', last_update_time=1791927161610, lifecycle_stage='active', name='/Users/labuser17579966_1791918748@vocareum.com/oai-sa-sdk-training', tags={'mlflow.experiment.sourceName': '/Users/labuser17579966_1791918748@vocareum.com/oai-sa-sdk-training',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'labuser17579966_1791918748@vocareum.com',
 'mlflow.ownerId': '74073739268346'}, trace_location=<UnityCatalog: full_annotations_table_name='labuser17579966_1791918748.agent_training.oai_sa_otel_annotations', full_otel_logs_table_name='labuser17579966_1791918748.agent_training.oai_sa_otel_logs', full_otel_spans_table_name='labuser17579966_1791918748.agent_training.oai_sa_otel_spans', full_table_prefix='labuser17579966_1791918748.agent_training.oai_sa', schema_location='labuser17579966_1791918748.ag

3. Navigate to Unity Catalog and see that we now have multiple tables and views with the prefix we set earlier, `oai_sa` ("OpenAI Single Agent"). These tables are empty since we haven't made any calls with our agent. Let's stress test the agent with and without MCP tools. 

### D3. Tracing without MCP

With autologging enabled, the next agent run will produce a full execution trace, which we will inspect in a moment. The trace captures the LLM reasoning step, the tool call to `classify_price_tier`, and the final response generation.

In [0]:
traced_query = "I found a listing for $225/night in the Mission district. Is that a fair price?"
result = await Runner.run(data_analyst_agent, traced_query)
print(f"User:     {traced_query}")
print(f"Response: {result.final_output}")

User:     I found a listing for $225/night in the Mission district. Is that a fair price?
Response: $225/night in SF (Mission) falls into the **Premium** tier, around the **50–75th percentile** of the market.

That means it’s **above average** for SF overall—**fair only if** the place has strong value drivers (e.g., entire unit vs. private room, modern/large space, great reviews, amenities like parking/laundry, or peak dates). If it’s a basic/private-room listing, it’s likely **priced high**.


Trace(trace_id=trace:/labuser17579966_1791918748.agent_training.oai_sa/dc52ac35479fe364baf0623eb26d23b2)

Navigate to the `oai_sa_otel_spans` table and click on the **Sample Data** tab. In the **name** column you will see `data_analyst_mcp_execution` and `AsyncDatabricksCompletions` This course does not go into details on spans and OTel concepts, but you can read more here ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/trace-unity-catalog)).

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Why Two Traces?</strong>
<p style="margin: 8px 0 0 0; color: #333;">
You may notice that a single agent query produces <strong>two separate traces</strong> in MLflow. This is expected behavior with <code>mlflow.openai.autolog()</code>. Because autologging patches at the <strong>OpenAI client level</strong>, each individual API call becomes its own top-level trace. A single <code>Runner.run()</code> call typically makes two underlying LLM calls:
</p>
<ol style="margin: 8px 0 0 16px; color: #333;">
<li><strong>First LLM call</strong>: The model receives the user query, reasons about it, and decides to invoke the <code>classify_price_tier</code> tool.</li>
<li><strong>Second LLM call</strong>: After the tool returns its result, the model generates the final natural-language response.</li>
</ol>
<p style="margin: 8px 0 0 0; color: #333;">
Without a parent span grouping them, each call appears as a separate trace. The next section solves this by wrapping the full agent run in <code>@mlflow.trace()</code>, which concatenates all spans into <strong>one unified trace</strong>.
</p>
</div>
</div>
</div>


### D4. Tracing with MCP

In a Databricks App deployment, the `McpServer` from `databricks_openai.agents` is used as an async context manager with the `Agent()` class.

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:20px 0;flex-wrap:wrap;font-family:system-ui,sans-serif;">
  <div style="background:#F9F7F4;border:2px solid #1B5162;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">@function_tool<br/><span style="font-weight:400;font-size:12px;">Local only</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#2574B5;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">UC Function<br/><span style="font-weight:400;font-size:12px;">Governed + reusable</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#02A36F;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">MCP Server<br/><span style="font-weight:400;font-size:12px;">Agent-accessible + standardized</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#FF5F46;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">Agent<br/><span style="font-weight:400;font-size:12px;">Ready for testing</span></div>
</div>

This is the recommended progression: start with `@function_tool` for rapid prototyping, register to UC for governance, then expose through MCP for production agent access.

1. Run the next cell to expose only `classify_price_uc` to the agent. 

In [0]:
from databricks_openai.agents import McpServer

mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="classify_price_uc",
    timeout=60.0,
)

2. **Redefine our agent** from before, but this time we pass `mcp_servers=[mcp_server]` instead of `tools`. The agent now discovers its tools at runtime from the UC function MCP server rather than from inline `@function_tool` definitions.

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
Why do we need @mlflow.trace, async def, and async?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">
<ul style="margin: 0; padding-left: 20px;">
<li><strong><code>@mlflow.trace(...)</code></strong>: Without this, the MCP connection, LLM call, and tool call each produce separate traces. Wrapping them in a single decorated function concatenates all spans into one unified trace.</li>
<li><strong><code>async def</code></strong>: <code>Runner.run()</code> is async (it makes HTTP calls to the model endpoint), and <code>McpServer</code> is an async context manager. Both require <code>await</code>, which means the enclosing function must be <code>async def</code>.</li>
<li><strong><code>async with mcp_server:</code></strong>: Opens the HTTP connection to the Databricks Managed MCP server before the agent runs, and closes it after. The agent calls <code>tools/list</code> and <code>tools/call</code> over this connection during execution.</li>
</ul>
</div>
</details>

In [0]:
data_analyst_mcp_agent_update = Agent(
    name="Data Analyst",
    instructions="You are a data analyst that helps users understand SF Airbnb listing prices. Use the classify_price_tier tool to categorize prices into market tiers. Provide clear, concise answers.",
    model=llm_endpoint,  # the same model service as the first agent
    mcp_servers=[mcp_server], # <---- used to be tools exposed via @function_tool
)

# Traced wrapper that concatenates all spans into a single trace
@mlflow.trace(span_type="AGENT", name="data_analyst_mcp_execution")
async def run_data_analyst_mcp(question: str) -> str:
    """Run the Data Analyst agent with MCP tools as a single traced operation."""
    async with mcp_server:
        result = await Runner.run(data_analyst_mcp_agent_update, question)
        return result.final_output

In [0]:
response = await run_data_analyst_mcp(query_1)
print(f"User:     {query_1}")
print(f"Response: {response}")

User:     Is $150/night a good deal in SF?
Response: $150/night in San Francisco is **Mid-Range**, roughly in the **25–50th percentile** of the SF Airbnb market—**around the market median**.

**Is it a good deal?** It’s **not especially cheap**, but it’s **reasonable/fair** for SF. It becomes a *good* deal if it includes strong value factors (great location, entire place vs. private room, parking, in-unit laundry, low cleaning fees, flexible cancellation). If it’s a basic/private-room listing or has high added fees, it’s more “average” than a bargain.


Trace(trace_id=trace:/labuser17579966_1791918748.agent_training.oai_sa/0e4a4e6b6f7f292c864c3f100cf8bf65)

### D5. View MLflow Traces
Finally, now that we have some traces logged with MLflow, we can view the traces in either the **Experiments** UI or directly from the unified trace table stored in the schema `agent_training` with the table prefix `oai_sa`. 

#### Experiment UI
1. Navigate to **Experiments** in the left sidebar, right click, and select **Open Link in New Tab**. 
2. Search for the experiment **oai-sa-sdk-training**. There you will see traces under the **Traces** tab. Click any trace to see a detailed breakdown of tools used and model calls.
#### MLflow Unified Tables from Unity Catalog
1. Navigate to the **Catalog Explorer** and search for **oai_sa**. This surface 6 default tables that have been created. 
    - 4 tables: **oai_sa_otel_logs**, **oai_sa_otel_spans**, **oai_sa_otel_annotations**, and **oai_sa_otel_annotations**
    - 2 views: **oai_sa_trace_unified**, and **oai_sa_trace_metadata**
2. Click on the **_otel_spans** table and click on **Sample Data** and ask a question in the provided query box like _Which trace took the longest to run?_ This will automatically generate a sql query (you might be asked to grant auto-run, click accept). The answer will be returned to you.
3. Run the next cell to see logged traces here in the notebook. Notice the same information is captured in the column `spans`.

In [0]:
%sql
SELECT * FROM oai_sa_trace_unified

trace_id client_request_id request_time state execution_duration_ms request response trace_metadata tags spans assessments dc52ac35479fe364baf0623eb26d23b2 null 2026-10-13T21:39:31.217Z OK 3749.695239 I found a listing for $225/night in the Mission district. Is that a fair price? $225/night in SF (Mission) falls into the **Premium** tier, around the **50–75th percentile** of the market.

That means it’s **above average** for SF overall—**fair only if** the place has strong value drivers (e.g., entire unit vs. private room, modern/large space, great reviews, amenities like parking/laundry, or peak dates). If it’s a basic/private-room listing, it’s likely **priced high**. {"mlflow.databricks.notebook.commandID":"1001791921611608_6804429202521585713_ce0e90da8cd9485e805943135a760450","mlflow.databricks.notebookID":"3585981954521399","mlflow.databricks.notebookPath":"/Users/labuser17579966_1791918748@vocareum.com/building-agentic-applications-on-databricks-en_us-2.0.0/Building Agentic Applications on Databricks/05 Demo - Introduction to OpenAI Agents SDK on Databricks","mlflow.databricks.webappURL":"https://dbc-358760f8-bd5d.cloud.databricks.com","mlflow.databricks.workspaceID":"7474660350819601","mlflow.databricks.workspaceURL":"https://dbc-358760f8-bd5d.cloud.databricks.com","mlflow.source.name":"/Users/labuser17579966_1791918748@vocareum.com/building-agentic-applications-on-databricks-en_us-2.0.0/Building Agentic Applications on Databricks/05 Demo - Introduction to OpenAI Agents SDK on Databricks","mlflow.source.type":"NOTEBOOK","mlflow.trace.cost":"{\"input_cost\": 0.0012215, \"output_cost\": 0.0016939999999999998, \"total_cost\": 0.0029154999999999997}","mlflow.trace.sizeBytes":"27815","mlflow.trace.sizeStats":"{\"total_size_bytes\": 27815, \"num_spans\": 10, \"max\": 8135, \"p25\": 577, \"p50\": 1037, \"p75\": 2761}","mlflow.trace.tokenUsage":"{\"input_tokens\": 698, \"output_tokens\": 121, \"total_tokens\": 819, \"cache_read_input_tokens\": 0, \"cache_creation_input_tokens\": 0}","mlflow.traceInputs":"\"I found a listing for $225/night in the Mission district. Is that a fair price?\"","mlflow.traceOutputs":"\"$225/night in SF (Mission) falls into the **Premium** tier, around the **50–75th percentile** of the market.\\n\\nThat means it’s **above average** for SF overall—**fair only if** the place has strong value drivers (e.g., entire unit vs. private ro...","mlflow.trace_schema.version":"4","mlflow.user":"spark-4514bf8e-c32c-4337-885d-e6"} Map(mlflow.traceName -> AgentRunner.run) List(List(01a13734-d314-739a-9dbf-3ca438268e79, 2026-10-13T21:39:31.220Z, 2026-10-13, null, dc52ac35479fe364baf0623eb26d23b2, 420ec0d6f19e9200, null, 8b12e308be96a323, null, AsyncDatabricksCompletions, null, 1791927571220306237, 1791927572542114377, {"extra_body":null,"extra_headers":{"User-Agent":"Agents/Python 0.22.3"},"extra_query":null,"frequency_penalty":" ","max_tokens":" ","metadata":" ","mlflow.chat.tokenUsage":{"cache_read_input_tokens":0,"input_tokens":311,"output_tokens":20,"total_tokens":331},"mlflow.chat.tools":[{"function":{"description":"Classify an Airbnb listing price into a market tier based on SF pricing distribution.\n\nUses hardcoded SF Airbnb market thresholds to label a nightly price as\nBudget, Mid-Range, Premium, or Luxury, and returns the corresponding\npercentile band for context.\n\nUseful for questions like: 'Is $180/night a good deal in SF?',\n'What tier does this listing fall into?'\n\nReturns a JSON object with tier, percentile_band, and a short interpretation.","name":"classify_price_tier","parameters":{"additionalProperties":false,"properties":{"price":{"description":"Nightly listing price in USD.","type":"number"}},"required":["price"],"type":"object"}},"type":"function"}],"mlflow.llm.cost":{"input_cost":5.4425E-4,"output_cost":2.8E-4,"total_cost":8.2425E-4},"mlflow.llm.model":"gpt-5.2-2025-12-11","mlflow.message.format":"openai","mlflow.spanInputs":{"extra_body":null,"extra_headers":{"User-Agent":"Agents/Python 0.

## E. Conclusion

In this demo, you:

1. Configured the OpenAI Agents SDK to call a Unity Catalog model service through Unity Gateway
2. Defined a `classify_price_tier` tool using the `@function_tool` decorator to classify Airbnb prices into market tiers
3. Built a single Data Analyst agent and tested it with progressively complex pricing queries
4. Registered `classify_price_tier` as a Python UC function for governance and connected it to the agent via MCP
5. Enabled MLflow autologging and inspected execution traces capturing LLM calls, tool invocations, and response generation

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>